In [12]:
import string
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import re
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix, precision_score, recall_score
)
from sklearn.preprocessing import StandardScaler

In [2]:
url = "https://github.com/bozercavdar/kickstarter-project/releases/download/v1.0/kickstarter_data_OSF.csv"
local_path = "dataset.csv"

df = pd.read_csv(local_path)

# print(df.head())

In [7]:
selected_df = df[['uid', 'blurb', 'goal', 'state', 'usd_pledged', 'category']]
selected_df['success'] = selected_df['state'] == 'successful'
selected_df.head()

/tmp/ipykernel_4042/949761106.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_df['success'] = selected_df['state'] == 'successful'


,uid,blurb,goal,state,usd_pledged,category,success
0,1,This project is designed to help protect the e...,2500.0,failed,0.0,Music,False
1,2,Help us built a sustainable studio & eliminate...,25000.0,failed,1.0,Technology,False
2,3,"""If I paint something, I don't want to have to...",5000.0,failed,5.0,Art,False
3,4,Our free app will allow you pool reservations ...,12000.0,failed,0.0,Food,False
4,5,Prohibition themed Gastro Pub and After Dark S...,20000.0,failed,0.0,Food,False


In [122]:
lexicon = ["exclusive", "limited", "innovative", "support", "backer", "prototype", "goal", "community", "early", "reward", "stretch", "sustainable",
    "handcrafted", "transparent", "impact", "trusted", "journey", "experience", "creative", "inspired"]

words = ["exclusive","limited","innovative","support","backer","prototype","goal","community","early","reward","stretch","sustainable","handcrafted","transparent","impact","trusted","journey","experience","creative","inspired","authentic","premium","launch","milestone","vision","mission","upgrade","beta","mvp","kickstarter","funded","pledge","unlock","bonus","limited edition","prototype-ready","eco-friendly","backer-only","behind-the-scenes","devlog","open-source","validated","guaranteed","craftsmanship","collaboration","creator","story","roadmap"]

words = [
    "new",
    "book",
    "album",
    "help",
    "series",
    "story",
    "short",
    "time",
    "inspired",
    "featuring",
    "film",
    "art",
    "life",
    "world",
    "music",
    "love",
    "game",
    "need",
    "project",
    "make"
]

# words = [
#     "series",
#     "short",
#     "time",
#     "story",
#     "inspired",
#     "featuring"
# ]

negatively_distinctive_words = [
    "want",
    "create",
    "people",
    "food",
    "creating",
    "like",
    "support",     # often used in failed projects but less impactful
    "community",   # tends to be more generic in failed projects
    "early",       # overused promises without follow-through
    "bonus",       # seen in failed campaigns offering gimmicky perks
    "upgrade",
    "beta",
    "limited edition",
    "exclusive", 
    "pledge"
]

In [123]:
def compute_score(text, dictionary):
    """
    Returns the number of dictionary matches per 100 words.
    """
    words = re.findall(r"\b\w+\b", text)
    word_count = len(words) if len(words) > 0 else 1

    count = 0
    for term in dictionary:
        # If phrase, check as substring
        if " " in term:
            if term in text:
                count += 1
        else:
            # Single-word matching
            count += words.count(term)

    return (count / word_count) * 100  # density per 100 words

In [124]:
selected_df["positive_score"] = selected_df["blurb"].apply(lambda x: compute_score(x, words))
selected_df["negative_score"] = selected_df["blurb"].apply(lambda x: compute_score(x, negatively_distinctive_words))
selected_df

/tmp/ipykernel_4042/729085213.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_df["positive_score"] = selected_df["blurb"].apply(lambda x: compute_score(x, words))
/tmp/ipykernel_4042/729085213.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_df["negative_score"] = selected_df["blurb"].apply(lambda x: compute_score(x, negatively_distinctive_words))


,uid,blurb,goal,state,usd_pledged,category,success,readiness_score,score,positive_score,negative_score
0,1,This project is designed to help protect the e...,2500.0,failed,0.000000,Music,False,0.000000,13.333333,13.333333,0.000000
1,2,Help us built a sustainable studio & eliminate...,25000.0,failed,1.000000,Technology,False,5.263158,0.000000,0.000000,0.000000
2,3,"""If I paint something, I don't want to have to...",5000.0,failed,5.000000,Art,False,0.000000,0.000000,0.000000,4.000000
3,4,Our free app will allow you pool reservations ...,12000.0,failed,0.000000,Food,False,0.000000,4.166667,4.166667,0.000000
4,5,Prohibition themed Gastro Pub and After Dark S...,20000.0,failed,0.000000,Food,False,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
160002,197831,Rob Carter's comedy creation Christopher Bliss...,5000.0,successful,6649.501355,Film & Video,True,0.000000,8.000000,8.000000,0.000000
160003,197833,"A sitcom about misfit superfans, nostalgic for...",5000.0,successful,8814.332641,Film & Video,True,0.000000,9.523810,9.523810,0.000000
160004,197834,High school students making homemade masks to ...,1000.0,successful,5068.000000,web,True,0.000000,6.666667,6.666667,0.000000
160005,197836,Le Attrata: Flame and Tech Meet in a Fiery Ste...,25000.0,successful,26056.660000,Art,True,0.000000,0.000000,0.000000,0.000000


In [130]:
X = selected_df[["goal", "positive_score", "negative_score"]].copy()
y = selected_df["success"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=41, stratify=y
)

In [131]:
pd.concat([X_train.head(), y_train.head()], axis=1)

,goal,positive_score,negative_score,success
33758,3000.0,0.000000,4.0,False
91961,12000.0,0.000000,0.0,True
136131,17000.0,11.111111,0.0,True
114083,1.0,0.000000,0.0,True
26216,400.0,10.000000,0.0,False


In [132]:
logit = LogisticRegression(max_iter=500)
logit.fit(X_train, y_train)

y_pred = logit.predict(X_test)
y_pred_prob = logit.predict_proba(X_test)[:, 1]

In [133]:
y_pred.mean()

np.float64(0.8766118784242651)

In [134]:
print("\n=== Logistic Regression Performance ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_pred_prob))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Coefficients
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coef": logit.coef_[0]
})
print("\n=== Coefficients ===")
print(coef_df)


=== Logistic Regression Performance ===
Accuracy: 0.6150657250588505
Precision: 0.6140684410646388
Recall: 0.9202934681957404
AUC: 0.6250873069759155

Confusion Matrix:
 [[ 3685 16240]
 [ 2238 25840]]

=== Coefficients ===
          feature      coef
0            goal -0.000003
1  positive_score  0.030515
2  negative_score -0.097227
